# Case 1: SC-wise Channel Prediction (Same Bandwidth)
- Subcarrier 64개를 각각 독립 sample로 취급
- 각 sample: input (PAST_LEN=16, 2*N_RX=32), target (32,)
- Train/Val 분리: 시간 기준 75% / 25% (겹침 없음)
- Models: LSTM, LWM

# Import

In [ ]:
import os
import sys
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader

from deepverse import ParameterManager, Dataset
from lwm_model2 import lwm

# Setting
### CUDA 번호 확인 후 변경

In [ ]:
N_SCENES  = 7000
PAST_LEN  = 16
EPOCHS    = 200
BATCH     = 256
LR        = 1e-4

SPLIT_TAG = "split4"  # 결과 파일 구분용 태그

DEVICE = torch.device("cuda:1")
print(f"Device: {DEVICE} | Scenes: {N_SCENES} | Epochs: {EPOCHS} | Batch: {BATCH} | Split: {SPLIT_TAG} | Norm: global")


# Generate Dataset

In [3]:
scenarios_name = "DT31"
config_path    = f"scenarios/{scenarios_name}/param/config.m"

param_manager = ParameterManager(config_path)
param_manager.params["scenes"] = list(range(N_SCENES))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))  # subcarrier 64개 사용
param_manager.params["radar"]["enable"] = False

dataset = Dataset(param_manager)


Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (8.35s)


# Preprocessing

In [4]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]
    ch_obj = ue_obj[ue_idx] if isinstance(ue_obj, (list, tuple)) else ue_obj
    if hasattr(ch_obj, "coeffs"):
        coeffs = ch_obj.coeffs
        if coeffs is None:
            raise ValueError(f"OFDMChannel.coeffs is None at ue_idx={ue_idx} — num_paths may be 0")
        return coeffs
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]
    raise TypeError(f"Cannot get coeffs: {type(ue_obj)}, {type(ch_obj)}")

comm_frames = flatten_comm_frames(dataset.comm_dataset)
N = len(comm_frames)
print(f"Total comm frames: {N}")

sample = get_coeffs_from_frame(comm_frames[0])
print(f"Channel shape per frame: {sample.shape}")

assert sample.ndim == 3, f"Unexpected channel shape: {sample.shape}"

# 가능한 레이아웃: (N_RX, 1, N_SC) 또는 (1, N_RX, N_SC)
if sample.shape[1] == 1:
    CH_LAYOUT = "rx_tx_sc"   # (N_RX, N_TX=1, N_SC)
    N_RX = int(sample.shape[0])
    N_SC = int(sample.shape[2])
elif sample.shape[0] == 1:
    CH_LAYOUT = "tx_rx_sc"   # (N_TX=1, N_RX, N_SC)
    N_RX = int(sample.shape[1])
    N_SC = int(sample.shape[2])
else:
    raise ValueError(f"Unsupported channel layout: {sample.shape}")

F_SC = 2 * N_RX
print(f"Auto-detected -> layout={CH_LAYOUT}, N_RX={N_RX}, N_SC={N_SC}, F_SC={F_SC}")


Total comm frames: 1000
Channel shape per frame: (1, 16, 64)


## Precompute raw channel data
- shape: (N, N_SC, N_RX, 2) — last dim: real/imag

In [5]:
print("Precomputing raw channel data...")
raw_data = np.zeros((N, N_SC, N_RX, 2), dtype=np.float32)
for t in range(N):
    c = get_coeffs_from_frame(comm_frames[t])
    for sc in range(N_SC):
        if CH_LAYOUT == "rx_tx_sc":
            sc_vals = c[:, 0, sc]      # (N_RX,)
        else:
            sc_vals = c[0, :, sc]      # (N_RX,)
        if sc_vals.shape[0] != N_RX:
            raise ValueError(f"N_RX mismatch at t={t}, sc={sc}: got {sc_vals.shape[0]}, expected {N_RX}")
        raw_data[t, sc, :, 0] = sc_vals.real
        raw_data[t, sc, :, 1] = sc_vals.imag
print(f"raw_data shape: {raw_data.shape}")


Precomputing raw channel data...
raw_data shape: (1000, 64, 16, 2)


# Train/Val Split (시간 기준 75% / 25%)

In [6]:
valid_start = PAST_LEN - 1   # 15: input [0..15] 확보
valid_end   = N - 2          # 998: target=t+1 최대 N-1

all_t   = list(range(valid_start, valid_end + 1))   # [15..998], 984개
n_train = int(len(all_t) * 0.75)                    # 738

train_t = all_t[:n_train]   # [15..752]  →  target 최대: 753

# 겹침 방지: val 입력이 train target(최대 753)을 포함하지 않으려면
# val_t[0] - (PAST_LEN-1) > 753  →  val_t[0] >= 752 + PAST_LEN + 1 = 769
val_start_t = train_t[-1] + PAST_LEN + 1            # 769
val_t = list(range(val_start_t, valid_end + 1))     # [769..998]

print(f"Train t 범위: [{train_t[0]}, {train_t[-1]}]  →  frame 사용: [0, {train_t[-1]+1}]")
gap_start, gap_end = train_t[-1] + 2, val_start_t - PAST_LEN
gap_str = f"[{gap_start}, {gap_end}]" if gap_start <= gap_end else "없음 (연속)"
print(f"Gap 구간   : frames {gap_str}  (사용 안 함)")
print(f"Val   t 범위: [{val_t[0]}, {val_t[-1]}]  →  frame 사용: [{val_t[0]-PAST_LEN+1}, {val_t[-1]+1}]")
print(f"\nSamples  — train: {len(train_t)*N_SC:,} | val: {len(val_t)*N_SC:,}")
print(f"(train:val = {len(train_t)/len(val_t):.2f}:1)")


In [7]:
def sample_used_indices(t, past_len):
    input_idx = set(range(t - past_len + 1, t + 1))  # input window
    target_idx = {t + 1}                             # target
    return input_idx, target_idx

# train 전체에서 사용한 raw time index
train_inputs = set()
train_targets = set()
for t in train_t:
    inp, tgt = sample_used_indices(t, PAST_LEN)
    train_inputs |= inp
    train_targets |= tgt

# val 전체에서 사용한 raw time index
val_inputs = set()
val_targets = set()
for t in val_t:
    inp, tgt = sample_used_indices(t, PAST_LEN)
    val_inputs |= inp
    val_targets |= tgt

print("train_t ∩ val_t =", set(train_t) & set(val_t))
print("train_inputs ∩ val_inputs =", sorted(train_inputs & val_inputs))
print("train_targets ∩ val_targets =", sorted(train_targets & val_targets))
print("train_targets ∩ val_inputs =", sorted(train_targets & val_inputs))
print("train_inputs ∩ val_targets =", sorted(train_inputs & val_targets))

train_t ∩ val_t = set()
train_inputs ∩ val_inputs = []
train_targets ∩ val_targets = []
train_targets ∩ val_inputs = []
train_inputs ∩ val_targets = []


# Global Min-Max Normalization (Global only)
- train 구간의 전체 채널(모든 SC, 모든 안테나) 기준 단일 min/max
- train 통계만 사용 (leakage 방지)


In [8]:
eps = 1e-16
channel_data = np.zeros((N, N_SC, F_SC), dtype=np.float32)
print("Computing global min/max from train set...")

train_real = raw_data[train_t, :, :, 0]   # (n_train, N_SC, N_RX)
train_imag = raw_data[train_t, :, :, 1]

r_min, r_max = float(train_real.min()), float(train_real.max())
i_min, i_max = float(train_imag.min()), float(train_imag.max())
print(f"Global scale  real: [{r_min:.6f}, {r_max:.6f}]")
print(f"              imag: [{i_min:.6f}, {i_max:.6f}]")

channel_data[:, :, :N_RX] = (raw_data[:, :, :, 0] - r_min) / max(r_max - r_min, eps)
channel_data[:, :, N_RX:] = (raw_data[:, :, :, 1] - i_min) / max(i_max - i_min, eps)

# CPU에 두고 학습 루프에서 .to(DEVICE) 처리 (DataLoader + CUDA tensor 호환성 문제 방지)
channel_tensor = torch.from_numpy(channel_data)  # (N, N_SC, F_SC), CPU
print(f"channel_tensor: {channel_tensor.shape} on {channel_tensor.device}")


Computing global min/max from train set...
Global scale  real: [-0.000001, 0.000002]
              imag: [-0.000002, 0.000002]
channel_tensor: torch.Size([1000, 64, 32]) on cpu


# Dataset 구현
- 각 sample = (time_window, SC) 쌍
- Input: (PAST_LEN=16, F_SC=32)
- Target: (F_SC=32,)

In [9]:
class SCwiseDataset(TorchDataset):
    def __init__(self, channel_tensor, sc_list, time_indices, past_len=16):
        self.ch       = channel_tensor   # (N, N_SC, F_SC) on CPU
        self.past_len = past_len
        self.samples  = [(t, sc) for t in time_indices for sc in sc_list]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        t, sc = self.samples[idx]
        ch_past = self.ch[t - self.past_len + 1 : t + 1, sc, :]  # (16, 32)
        target  = self.ch[t + 1, sc, :]                           # (32,)
        return ch_past, target

all_sc        = list(range(N_SC))
train_dataset = SCwiseDataset(channel_tensor, all_sc, train_t, PAST_LEN)
val_dataset   = SCwiseDataset(channel_tensor, all_sc, val_t,   PAST_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

ch_s, y_s = next(iter(train_loader))
print(f"Batch - ch: {ch_s.shape}, y: {y_s.shape}")   # (B, 16, 32), (B, 32)

Batch - ch: torch.Size([256, 16, 32]), y: torch.Size([256, 32])


# Model 정의
## LSTM

In [10]:
class LSTMForecaster(nn.Module):
    """Input: (B, T, F_in=32)  ->  Output: (B, F_out=32)"""
    def __init__(self, F_in, F_out, hidden=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=F_in, hidden_size=hidden, num_layers=num_layers,
            batch_first=True, dropout=(dropout if num_layers > 1 else 0.0)
        )
        self.head = nn.Linear(hidden, F_out)

    def forward(self, ch):
        out, _ = self.lstm(ch)
        return self.head(out[:, -1, :])

## LWM

In [ ]:
class LWMForecaster(nn.Module):
    """LWM backbone embedding+layers 직접 호출
    Input: (B, T, F_in=32)  ->  Output: (B, F_out=32)
    """
    def __init__(self, base_lwm, F_in, F_out, freeze_backbone=False):
        super().__init__()
        self.base    = base_lwm
        n_dim        = self.base.embedding.element_length  # lwm_model2: 16
        d_model      = self.base.embedding.d_model          # lwm_model2: 64
        self.in_proj = nn.Linear(F_in, n_dim)
        self.head    = nn.Linear(d_model, F_out)
        if freeze_backbone:
            for p in self.base.parameters():
                p.requires_grad = False

    def forward(self, ch):
        x   = self.in_proj(ch)                # (B, T, n_dim=16)
        out = self.base.embedding(x)           # (B, T, d_model=64)
        for layer in self.base.layers:
            out, _ = layer(out)
        return self.head(out[:, -1, :])        # (B, F_out=32)


# NMSE(dB)

In [12]:
@torch.no_grad()
def nmse_components(yhat, y):
    """배치 단위 분자/분모 합산값 반환 (global NMSE 계산용)"""
    num = torch.sum((yhat - y) ** 2).item()
    den = torch.sum(y ** 2).item()
    return num, den

def nmse_db(num_total, den_total, eps=1e-16):
    """Global NMSE(dB) = 10·log10( Σ||e||² / Σ||y||² )"""
    return 10.0 * np.log10(max(num_total / max(den_total, eps), eps))


# Train / Eval 함수

In [13]:
def train_one_epoch(model, loader, optimizer, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_num, total_den = 0.0, 0.0
    n = 0
    for ch, y in loader:
        ch, y = ch.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        yhat = model(ch)
        loss = F.mse_loss(yhat, y)
        loss.backward()
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
        num, den = nmse_components(yhat.detach(), y)
        total_num += num
        total_den += den
        n += 1
    return total_loss / max(n, 1), nmse_db(total_num, total_den)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_num, total_den = 0.0, 0.0
    n = 0
    for ch, y in loader:
        ch, y = ch.to(DEVICE), y.to(DEVICE)
        yhat = model(ch)
        loss = F.mse_loss(yhat, y)
        total_loss += loss.item()
        num, den = nmse_components(yhat, y)
        total_num += num
        total_den += den
        n += 1
    return total_loss / max(n, 1), nmse_db(total_num, total_den)

# 학습 루프

In [ ]:
def run_experiment(model_name, model, log_path, ckpt_path):
    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*55}")
    print(f" {model_name} | total={total_p:,} | trainable={trainable_p:,}")
    print(f"{'='*55}")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val, best_nmse, best_epoch = float("inf"), None, None
    t0_total = time.time()

    with open(log_path, "w", encoding="utf-8") as f:
        header = (
            f"=== {model_name} | Case1 SC-wise | scenes={N_SCENES} | "
            f"train={len(train_dataset):,} | val={len(val_dataset):,} ===\n"
        )
        print(header.strip()); f.write(header)

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()
            tr_loss, tr_nmse = train_one_epoch(model, train_loader, optimizer)
            va_loss, va_nmse = evaluate(model, val_loader)
            scheduler.step()

            line = (
                f"[{epoch:03d}/{EPOCHS}] "
                f"train loss={tr_loss:.6f} nmse={tr_nmse:.4f}dB | "
                f"val loss={va_loss:.6f} nmse={va_nmse:.4f}dB | "
                f"{time.time()-t0:.1f}s"
            )
            print(line); f.write(line + "\n"); f.flush()

            if va_loss < best_val:
                best_val, best_nmse, best_epoch = va_loss, va_nmse, epoch
                torch.save({
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "F_in": F_SC, "F_out": F_SC,
                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_nmse,
                    "n_scenes": N_SCENES,
                }, ckpt_path)
                ckpt_line = f"  -> saved {ckpt_path} | best@{epoch}: val nmse={best_nmse:.4f}dB"
                print(ckpt_line); f.write(ckpt_line + "\n"); f.flush()

        total = time.time() - t0_total
        h, m, s = int(total//3600), int((total%3600)//60), total%60
        summary = (
            f"\n=== Done === {h}h {m}m {s:.0f}s | "
            f"Best epoch {best_epoch}: val loss={best_val:.6f} nmse={best_nmse:.4f}dB"
        )
        print(summary); f.write(summary + "\n")

    return best_val, best_nmse


In [ ]:
# Copy-last baseline: 마지막 입력 시점을 그대로 예측
@torch.no_grad()
def copy_last_nmse(loader):
    total_num, total_den = 0.0, 0.0
    for ch, y in loader:
        yhat = ch[:, -1, :].to(DEVICE)   # 마지막 입력 step 복사
        num, den = nmse_components(yhat, y.to(DEVICE))
        total_num += num
        total_den += den
    return nmse_db(total_num, total_den)

baseline_val_nmse  = copy_last_nmse(val_loader)
baseline_train_nmse = copy_last_nmse(train_loader)
print(f"Copy-last baseline  train NMSE = {baseline_train_nmse:.4f} dB")
print(f"Copy-last baseline  val   NMSE = {baseline_val_nmse:.4f} dB")
print("(모델이 이 값보다 낮아야 의미 있는 예측)")


# LSTM 학습

In [15]:
lstm_model = LSTMForecaster(F_in=F_SC, F_out=F_SC, hidden=256, num_layers=2).to(DEVICE)
print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}")

with torch.no_grad():
    yhat = lstm_model(ch_s.to(DEVICE))
print(f"yhat: {yhat.shape}, y: {y_s.shape}")

LSTM params: 831,520
yhat: torch.Size([256, 32]), y: torch.Size([256, 32])


In [ ]:
lstm_loss, lstm_nmse = run_experiment(
    model_name="LSTM",
    model=lstm_model,
    log_path=f"case1_{SPLIT_TAG}_lstm_scene{N_SCENES}.txt",
    ckpt_path=f"case1_{SPLIT_TAG}_lstm_scene{N_SCENES}_best.pt",
)

# LWM 학습
## 3가지 변형: FromScratch / Finetune / Freeze

In [ ]:
PRETRAINED_CKPT = "model_weights.pth"

# 1) lwm_fromScratch: 사전학습 가중치 없이 랜덤 초기화
backbone_scratch = lwm().to(DEVICE)
lwm_fromScratch  = LWMForecaster(base_lwm=backbone_scratch, F_in=F_SC, F_out=F_SC, freeze_backbone=False).to(DEVICE)

# 2) lwm_finetune: 사전학습 가중치 로드, 백본 포함 전체 학습
backbone_finetune = lwm.from_pretrained(PRETRAINED_CKPT, device=DEVICE)
lwm_finetune      = LWMForecaster(base_lwm=backbone_finetune, F_in=F_SC, F_out=F_SC, freeze_backbone=False).to(DEVICE)

# 3) lwm_freeze: 사전학습 가중치 로드, 백본 동결 (in_proj / head 만 학습)
backbone_freeze = lwm.from_pretrained(PRETRAINED_CKPT, device=DEVICE)
lwm_freeze      = LWMForecaster(base_lwm=backbone_freeze, F_in=F_SC, F_out=F_SC, freeze_backbone=True).to(DEVICE)

for name, model in [("lwm_fromScratch", lwm_fromScratch),
                    ("lwm_finetune",    lwm_finetune),
                    ("lwm_freeze",      lwm_freeze)]:
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:20s}  total={total:>9,}  trainable={trainable:>9,}")

with torch.no_grad():
    yhat = lwm_fromScratch(ch_s.to(DEVICE))
print(f"\nyhat: {yhat.shape}, y: {y_s.shape}")

In [ ]:
scratch_loss, scratch_nmse = run_experiment(
    model_name="lwm_fromScratch",
    model=lwm_fromScratch,
    log_path=f"case1_{SPLIT_TAG}_lwm_fromScratch_scene{N_SCENES}.txt",
    ckpt_path=f"case1_{SPLIT_TAG}_lwm_fromScratch_scene{N_SCENES}_best.pt",
)

finetune_loss, finetune_nmse = run_experiment(
    model_name="lwm_finetune",
    model=lwm_finetune,
    log_path=f"case1_{SPLIT_TAG}_lwm_finetune_scene{N_SCENES}.txt",
    ckpt_path=f"case1_{SPLIT_TAG}_lwm_finetune_scene{N_SCENES}_best.pt",
)

freeze_loss, freeze_nmse = run_experiment(
    model_name="lwm_freeze",
    model=lwm_freeze,
    log_path=f"case1_{SPLIT_TAG}_lwm_freeze_scene{N_SCENES}.txt",
    ckpt_path=f"case1_{SPLIT_TAG}_lwm_freeze_scene{N_SCENES}_best.pt",
)

# 최종 결과 비교

In [ ]:
print(f"\n{'='*60}")
print(f" Final Results - Case 1 SC-wise | {SPLIT_TAG} | batch={BATCH} | scenes={N_SCENES}")
print(f"{'='*60}")
print(f"  LSTM            : best val NMSE = {lstm_nmse:.4f} dB")
print(f"  LWM fromScratch : best val NMSE = {scratch_nmse:.4f} dB")
print(f"  LWM finetune    : best val NMSE = {finetune_nmse:.4f} dB")
print(f"  LWM freeze      : best val NMSE = {freeze_nmse:.4f} dB")
print(f"{'='*60}")

# Debug 출력 (실제값/정규화값/타깃/예측)
- val sample 1개를 선택해서 입력 마지막 step, target, 모델 예측을 함께 출력


In [ ]:
# Debug sample index (0 ~ len(val_dataset)-1)
SAMPLE_IDX = 0
NUM_SHOW = min(4, N_RX)  # 출력할 Rx 개수

assert 0 <= SAMPLE_IDX < len(val_dataset), f"SAMPLE_IDX out of range: {SAMPLE_IDX}"

def split_ri(vec):
    """(2*N_RX,) -> (real[N_RX], imag[N_RX])"""
    vec = np.asarray(vec)
    return vec[:N_RX], vec[N_RX:]

def denorm_vec(vec_norm):
    """global min-max 역정규화"""
    r_norm, i_norm = split_ri(vec_norm)
    r = r_norm * (r_max - r_min) + r_min
    i = i_norm * (i_max - i_min) + i_min
    return r, i

def print_ri_block(title, r_vals, i_vals, n=4, sci=False):
    fmt = ".6e" if sci else ".6f"
    print(f"\n[{title}] (first {n} rx)")
    for k in range(n):
        r_str = f"{r_vals[k]: .6e}" if sci else f"{r_vals[k]: .6f}"
        i_str = f"{i_vals[k]: .6e}" if sci else f"{i_vals[k]: .6f}"
        print(f"  rx{k:02d}: real={r_str} | imag={i_str}")

# val_dataset sample -> (ch_past_norm, target_norm)
ch_past_norm, target_norm = val_dataset[SAMPLE_IDX]
t, sc = val_dataset.samples[SAMPLE_IDX]

# 입력 마지막 시점(t), 타깃 시점(t+1)의 실제(raw) 값
x_last_raw_r = raw_data[t, sc, :, 0]
x_last_raw_i = raw_data[t, sc, :, 1]
y_raw_r      = raw_data[t + 1, sc, :, 0]
y_raw_i      = raw_data[t + 1, sc, :, 1]

# 입력 마지막 시점의 정규화 값
x_last_norm_r, x_last_norm_i = split_ri(ch_past_norm[-1].numpy())
# 타깃 정규화 값
y_norm_r, y_norm_i = split_ri(target_norm.numpy())

# 모델 예측 (정규화 공간)
with torch.no_grad():
    x_in = ch_past_norm.unsqueeze(0).to(DEVICE)
    yhat_lstm_norm    = lstm_model(x_in).squeeze(0).cpu().numpy()
    yhat_scratch_norm = lwm_fromScratch(x_in).squeeze(0).cpu().numpy()
    yhat_finetune_norm= lwm_finetune(x_in).squeeze(0).cpu().numpy()
    yhat_freeze_norm  = lwm_freeze(x_in).squeeze(0).cpu().numpy()

lstm_norm_r,    lstm_norm_i    = split_ri(yhat_lstm_norm)
scratch_norm_r, scratch_norm_i = split_ri(yhat_scratch_norm)
finetune_norm_r,finetune_norm_i= split_ri(yhat_finetune_norm)
freeze_norm_r,  freeze_norm_i  = split_ri(yhat_freeze_norm)

# 모델 예측 (실제값 공간으로 역정규화)
lstm_raw_r,     lstm_raw_i     = denorm_vec(yhat_lstm_norm)
scratch_raw_r,  scratch_raw_i  = denorm_vec(yhat_scratch_norm)
finetune_raw_r, finetune_raw_i = denorm_vec(yhat_finetune_norm)
freeze_raw_r,   freeze_raw_i   = denorm_vec(yhat_freeze_norm)

print(f"Selected val sample idx={SAMPLE_IDX}, time t={t}, sc={sc}")
print(f"Input window: [{t-PAST_LEN+1}..{t}] -> Target: {t+1}")

print_ri_block("Input(last step) RAW",  x_last_raw_r, x_last_raw_i, NUM_SHOW, sci=True)
print_ri_block("Input(last step) NORM", x_last_norm_r, x_last_norm_i, NUM_SHOW)
print_ri_block("Target RAW",  y_raw_r, y_raw_i, NUM_SHOW, sci=True)
print_ri_block("Target NORM", y_norm_r, y_norm_i, NUM_SHOW)

print_ri_block("LSTM       Pred NORM", lstm_norm_r,    lstm_norm_i,    NUM_SHOW)
print_ri_block("LSTM       Pred RAW",  lstm_raw_r,     lstm_raw_i,     NUM_SHOW, sci=True)
print_ri_block("LWM Scratch Pred NORM",scratch_norm_r, scratch_norm_i, NUM_SHOW)
print_ri_block("LWM Scratch Pred RAW", scratch_raw_r,  scratch_raw_i,  NUM_SHOW, sci=True)
print_ri_block("LWM Finetune Pred NORM",finetune_norm_r,finetune_norm_i,NUM_SHOW)
print_ri_block("LWM Finetune Pred RAW", finetune_raw_r, finetune_raw_i, NUM_SHOW, sci=True)
print_ri_block("LWM Freeze  Pred NORM",freeze_norm_r,  freeze_norm_i,  NUM_SHOW)
print_ri_block("LWM Freeze  Pred RAW", freeze_raw_r,   freeze_raw_i,   NUM_SHOW, sci=True)
